# 20. Калибровка Hierarchical Span NER

Порог выбирается только на validation. Дополнительно сохраняются validation-предсказания, необходимые для калибровки полного NER→RE pipeline.

In [ ]:
from pathlib import Path
import os, runpy
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
if not PROJECT_DIR.exists(): PROJECT_DIR = Path.cwd()
os.environ['HF_HOME'] = '/content/huggingface_cache'
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)


In [ ]:
from rurebus_ie.training import calibrate_hierarchical_span_threshold_experiment, evaluate_hierarchical_span_ner_experiment
CONFIG = PROJECT_DIR / 'configs/experiments/hierarchical_span_ner_global_v1.yaml'
THRESHOLDS = [round(0.30 + step * 0.01, 2) for step in range(69)]
calibration = calibrate_hierarchical_span_threshold_experiment(CONFIG, thresholds=THRESHOLDS, project_root=PROJECT_DIR)
BEST_THRESHOLD = calibration.best_threshold
print(f'Лучший NER threshold: {BEST_THRESHOLD:.2f}; validation micro-F1={calibration.best_metrics.micro_f1:.6f}')
validation = evaluate_hierarchical_span_ner_experiment(CONFIG, split_key='validation', project_root=PROJECT_DIR, confidence_threshold_override=BEST_THRESHOLD, artifact_prefix='calibrated_validation')
print('Validation predictions сохранены для pipeline calibration.')
